# Assignment 1B - Part A Step 1

## Domain Data Collection & Cleaning

**Chosen domain:** Medical & Clinical Literature  
**Chosen base model:** `microsoft/biogpt-large`  

This notebook section builds the reusable cleaned corpus for the rest of the assignment. The same cleaned text files created here will be reused later for instruction dataset construction, fine-tuning, and inference benchmarking.

**Tokenizer rule for later parts:** whenever tokenization is needed, use the tokenizer paired with BioGPT-Large: `AutoTokenizer.from_pretrained("microsoft/biogpt-large")`. We do not mix tokenizers across models because every tokenizer has its own vocabulary and token IDs.

## What this step expects

Place at least **5 medical or clinical PDF files** inside a folder named `raw_pdfs/` before running the extraction cells.

The pipeline below will:

1. Read PDFs from `raw_pdfs/`.
2. Extract text page-by-page using PyMuPDF.
3. Keep only English documents using `langdetect`.
4. Remove repeated headers/footers and exact duplicate pages using hashing.
5. Remove documents that are too short to be useful.
6. Remove near-duplicate documents using word-shingle Jaccard similarity.
7. Save the final cleaned `.txt` files to `domain_corpus/`.
8. Print corpus statistics before and after every cleaning step.

In [18]:
# Run this once in a fresh Colab/Jupyter environment.
# PyMuPDF is imported as `fitz` and is used to read PDF pages.
# langdetect is used to keep only English documents.
# tqdm gives progress bars while processing multiple PDFs.
%pip install -q pymupdf langdetect tqdm

Note: you may need to restart the kernel to use updated packages.


In [19]:
# -----------------------------
# Part A configuration
# -----------------------------

from pathlib import Path
import hashlib
import json
import math
import re
import unicodedata
from collections import Counter

import fitz  # PyMuPDF
from langdetect import DetectorFactory, LangDetectException, detect_langs
from tqdm.auto import tqdm

# langdetect has a small random component. Setting the seed makes repeated runs stable.
DetectorFactory.seed = 42

# We store the chosen domain/model once so every later part can reuse the same values.
DOMAIN = 'Medical & Clinical Literature'
MODEL_ID = 'microsoft/biogpt-large'

# Input folder: put your original PDFs here.
RAW_PDF_DIR = Path('raw_pdfs')

# Final assignment output folder: cleaned .txt files are saved here.
OUTPUT_DIR = Path('domain_corpus')

# Work folder: intermediate files and JSON reports are saved here for audit/debugging.
WORK_DIR = Path('part_a_work')
RAW_EXTRACTED_DIR = WORK_DIR / '01_extracted_raw'

# Length filter justification:
# A document with fewer than 5,000 extracted characters is usually too small for useful
# domain adaptation. It may only contain an abstract, a short handout, or a mostly blank PDF.
# 5,000 characters is roughly 800-1,000 English words, which is enough to contain several
# clinical/medical concepts and produce meaningful instruction examples later.
MIN_DOCUMENT_CHARACTERS = 5_000

# Language filter: retain only documents whose top detected language is English with high confidence.
ENGLISH_CONFIDENCE_THRESHOLD = 0.85

# Near-duplicate document filter:
# We compare sets of 5-word chunks. If two documents share at least 85% of those chunks,
# we treat the shorter one as a near-duplicate and remove it.
SHINGLE_SIZE = 5
DOC_DUPLICATE_JACCARD_THRESHOLD = 0.85

# Create folders if they do not exist yet.
for directory in [RAW_PDF_DIR, OUTPUT_DIR, WORK_DIR, RAW_EXTRACTED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# These variables collect statistics and drop reasons across the cleaning steps.
stage_stats = []
drop_logs = {
    'unreadable_or_empty_pdfs': [],
    'language_filter': [],
    'page_or_section_deduplication': [],
    'length_filter': [],
    'document_near_duplicate_filter': [],
}

print(f'Domain: {DOMAIN}')
print(f'Model selected for later fine-tuning: {MODEL_ID}')
print(f'Put source PDFs in: {RAW_PDF_DIR.resolve()}')
print(f'Final cleaned corpus will be written to: {OUTPUT_DIR.resolve()}')

Domain: Medical & Clinical Literature
Model selected for later fine-tuning: microsoft/biogpt-large
Put source PDFs in: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/raw_pdfs
Final cleaned corpus will be written to: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/domain_corpus


## Helper functions

These functions keep the main pipeline readable. Each helper does one small job: clean text, count words, rebuild a document from pages, detect language, print tables, or compute duplicate similarity.

In [20]:
def safe_stem(filename):
    """Create a safe file-name stem from an original PDF name."""
    stem = Path(filename).stem.lower()
    stem = re.sub(r'[^a-z0-9]+', '_', stem)
    return stem.strip('_') or 'document'


def clean_page_text(raw_text):
    """Basic cleanup for text extracted from one PDF page."""
    if not raw_text:
        return ''

    # Normalize unicode so visually similar characters are represented consistently.
    text = unicodedata.normalize('NFKC', raw_text)

    # PDF extraction often splits words at line breaks: "clin-\nical".
    # This joins them back into "clinical".
    text = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', text)

    # Make line endings consistent across operating systems.
    text = text.replace('\r', '\n')

    # Collapse repeated spaces/tabs, but keep newlines because they help preserve paragraphs.
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n[ \t]+', '\n', text)

    # Collapse excessive blank lines.
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


def count_words(text):
    """Count simple word-like tokens for corpus statistics."""
    return len(re.findall(r'\b\w+\b', text))


def rebuild_document_text(pages, page_numbers):
    """Join page texts into one document while keeping source page markers."""
    sections = []
    for page_number, page_text in zip(page_numbers, pages):
        sections.append(f'[Source page {page_number}]\n{page_text}')
    return '\n\n'.join(sections).strip()


def detect_document_language(text):
    """Detect the most likely language using langdetect."""
    # langdetect only needs a sample. Using the first 20k characters keeps it fast.
    sample = text[:20_000]

    if len(sample.strip()) < 200:
        return 'unknown', 0.0

    try:
        detected = detect_langs(sample)
    except LangDetectException:
        return 'unknown', 0.0

    best = detected[0]
    return best.lang, float(best.prob)


def normalize_for_duplicate_detection(text):
    """Normalize text before hashing or shingling for duplicate checks."""
    text = unicodedata.normalize('NFKC', text).lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def add_stage(stage, records=None, pdf_count=0, page_count=0, note=''):
    """Save a snapshot of corpus size after each major step."""
    if records is None:
        row = {
            'stage': stage,
            'source_documents': pdf_count,
            'txt_files': 0,
            'pages': page_count,
            'characters': 0,
            'words': 0,
            'note': note,
        }
    else:
        row = {
            'stage': stage,
            'source_documents': len(records),
            'txt_files': len(records),
            'pages': sum(len(record['pages']) for record in records),
            'characters': sum(len(record['text']) for record in records),
            'words': sum(count_words(record['text']) for record in records),
            'note': note,
        }

    stage_stats.append(row)
    return row


def print_table(rows, columns):
    """Print a small plain-text table without requiring pandas."""
    if not rows:
        print('No rows to show.')
        return

    widths = {}
    for column in columns:
        widths[column] = max([len(column)] + [len(str(row.get(column, ''))) for row in rows])

    header = ' | '.join(column.ljust(widths[column]) for column in columns)
    divider = '-+-'.join('-' * widths[column] for column in columns)
    print(header)
    print(divider)

    for row in rows:
        print(' | '.join(str(row.get(column, '')).ljust(widths[column]) for column in columns))

## 1. Discover PDFs and extract text page-by-page

We first count the raw PDFs and their pages. Then we extract every readable page separately. Keeping page boundaries helps us later remove repeated pages and report page-level statistics.

In [21]:
# Restart statistics if this cell is rerun.
stage_stats = []
for key in drop_logs:
    drop_logs[key] = []

pdf_paths = sorted(RAW_PDF_DIR.glob('*.pdf'))

if len(pdf_paths) < 5:
    raise ValueError(
        f'Expected at least 5 PDFs in {RAW_PDF_DIR.resolve()}, but found {len(pdf_paths)}. '
        'Add medical/clinical PDF files to raw_pdfs/ and rerun this cell.'
    )

raw_pdf_page_counts = {}
for pdf_path in pdf_paths:
    try:
        with fitz.open(pdf_path) as pdf:
            raw_pdf_page_counts[pdf_path.name] = pdf.page_count
    except Exception as error:
        # The extraction loop below will also skip unreadable PDFs, but we log them here for reporting.
        raw_pdf_page_counts[pdf_path.name] = 0
        drop_logs['unreadable_or_empty_pdfs'].append({
            'source_pdf': pdf_path.name,
            'reason': f'Could not open PDF: {error}',
        })

add_stage(
    stage='0. Raw PDFs collected',
    records=None,
    pdf_count=len(pdf_paths),
    page_count=sum(raw_pdf_page_counts.values()),
    note='Original PDFs placed in raw_pdfs/',
)

extracted_records = []

for source_index, pdf_path in enumerate(tqdm(pdf_paths, desc='Extracting PDFs'), start=1):
    try:
        with fitz.open(pdf_path) as pdf:
            page_texts = []
            page_numbers = []

            for page_index, page in enumerate(pdf, start=1):
                page_text = clean_page_text(page.get_text('text'))

                # Very tiny pages are usually blank pages, scanned-image pages with no OCR,
                # or pages that contain only a page number. We ignore them at extraction time.
                if len(page_text) >= 50:
                    page_texts.append(page_text)
                    page_numbers.append(page_index)

    except Exception as error:
        drop_logs['unreadable_or_empty_pdfs'].append({
            'source_pdf': pdf_path.name,
            'reason': f'Could not extract text: {error}',
        })
        continue

    if not page_texts:
        drop_logs['unreadable_or_empty_pdfs'].append({
            'source_pdf': pdf_path.name,
            'reason': 'No readable text found. The PDF may be scanned and may need OCR.',
        })
        continue

    document_text = rebuild_document_text(page_texts, page_numbers)
    record = {
        'source_index': source_index,
        'source_pdf': pdf_path.name,
        'source_path': str(pdf_path),
        'raw_page_count': raw_pdf_page_counts.get(pdf_path.name, len(page_texts)),
        'page_numbers': page_numbers,
        'pages': page_texts,
        'text': document_text,
    }
    extracted_records.append(record)

    # Save the raw extracted version for audit. The final cleaned version is saved later.
    raw_output_name = f'{source_index:02d}_{safe_stem(pdf_path.name)}.txt'
    (RAW_EXTRACTED_DIR / raw_output_name).write_text(document_text, encoding='utf-8')

add_stage(
    stage='1. After text extraction',
    records=extracted_records,
    note='Readable page text extracted with PyMuPDF',
)

print(f'PDFs found: {len(pdf_paths)}')
print(f'PDFs with readable extracted text: {len(extracted_records)}')
print(f'Raw extracted text files saved in: {RAW_EXTRACTED_DIR.resolve()}')

Extracting PDFs: 100%|██████████| 10/10 [00:00<00:00, 33.45it/s]

PDFs found: 10
PDFs with readable extracted text: 10
Raw extracted text files saved in: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/part_a_work/01_extracted_raw


## 2. Language filter

The assignment asks us to retain only English-language documents. We use `langdetect`, keep documents whose top language is `en`, and require confidence of at least `0.85`.

In [22]:
english_records = []

for record in tqdm(extracted_records, desc='Detecting language'):
    language, confidence = detect_document_language(record['text'])

    updated_record = record.copy()
    updated_record['detected_language'] = language
    updated_record['language_confidence'] = round(confidence, 4)

    if language == 'en' and confidence >= ENGLISH_CONFIDENCE_THRESHOLD:
        english_records.append(updated_record)
    else:
        drop_logs['language_filter'].append({
            'source_pdf': record['source_pdf'],
            'detected_language': language,
            'confidence': round(confidence, 4),
            'reason': 'Top detected language was not confident English.',
        })

add_stage(
    stage='2. After English language filter',
    records=english_records,
    note=f'Kept lang=en with confidence >= {ENGLISH_CONFIDENCE_THRESHOLD}',
)

print(f'Documents kept after language filtering: {len(english_records)}')
print(f'Documents removed by language filtering: {len(drop_logs["language_filter"])}')

Detecting language: 100%|██████████| 10/10 [00:00<00:00, 27.83it/s]

Documents kept after language filtering: 10
Documents removed by language filtering: 0


## 3. Deduplication of repeated pages and repeated sections

This step handles two common PDF problems:

- Repeated headers/footers such as journal names, copyright text, or page banners.
- Exact duplicate pages across documents.

For repeated pages, we normalize each page and compute a SHA-256 hash. If the same normalized page hash appears again, we remove the later copy.

In [23]:
def normalize_line_for_repetition(line):
    """Normalize one line before checking whether it is a repeated header/footer."""
    normalized = unicodedata.normalize('NFKC', line).lower()
    normalized = re.sub(r'\d+', '#', normalized)
    normalized = re.sub(r'\s+', ' ', normalized).strip()

    # Very short lines are often page numbers. Very long lines are usually real content.
    if len(normalized) < 6 or len(normalized) > 120:
        return ''

    return normalized


def remove_repeated_lines_from_pages(pages):
    """Remove short lines repeated on many pages of the same document."""
    if len(pages) < 3:
        return pages, set(), 0

    line_counter = Counter()

    for page_text in pages:
        normalized_lines_on_page = set()
        for line in page_text.splitlines():
            normalized_line = normalize_line_for_repetition(line)
            if normalized_line:
                normalized_lines_on_page.add(normalized_line)
        line_counter.update(normalized_lines_on_page)

    # A line repeated on at least 60% of pages, and at least 3 pages, is likely a header/footer.
    repeat_threshold = max(3, math.ceil(len(pages) * 0.60))
    repeated_lines = {
        line for line, count in line_counter.items()
        if count >= repeat_threshold
    }

    cleaned_pages = []
    removed_line_count = 0

    for page_text in pages:
        kept_lines = []
        for line in page_text.splitlines():
            normalized_line = normalize_line_for_repetition(line)
            if normalized_line and normalized_line in repeated_lines:
                removed_line_count += 1
                continue
            kept_lines.append(line)

        cleaned_pages.append(clean_page_text('\n'.join(kept_lines)))

    return cleaned_pages, repeated_lines, removed_line_count


def deduplicate_pages_and_repeated_sections(records):
    """Remove repeated page/section noise and return cleaned records."""
    seen_page_hashes = set()
    cleaned_records = []

    for record in records:
        pages_without_repeated_lines, repeated_lines, removed_line_count = remove_repeated_lines_from_pages(record['pages'])

        kept_pages = []
        kept_page_numbers = []
        duplicate_pages_removed = 0

        for page_number, page_text in zip(record['page_numbers'], pages_without_repeated_lines):
            normalized_page = normalize_for_duplicate_detection(page_text)

            # Tiny pages are not reliable for hashing, so we keep them and let the length filter decide later.
            if len(normalized_page) < 100:
                kept_pages.append(page_text)
                kept_page_numbers.append(page_number)
                continue

            page_hash = hashlib.sha256(normalized_page.encode('utf-8')).hexdigest()

            if page_hash in seen_page_hashes:
                duplicate_pages_removed += 1
                continue

            seen_page_hashes.add(page_hash)
            kept_pages.append(page_text)
            kept_page_numbers.append(page_number)

        if not kept_pages:
            drop_logs['page_or_section_deduplication'].append({
                'source_pdf': record['source_pdf'],
                'reason': 'All pages were duplicate pages already seen in the corpus.',
            })
            continue

        updated_record = record.copy()
        updated_record['pages'] = kept_pages
        updated_record['page_numbers'] = kept_page_numbers
        updated_record['text'] = rebuild_document_text(kept_pages, kept_page_numbers)
        updated_record['repeated_line_patterns_removed'] = len(repeated_lines)
        updated_record['repeated_line_instances_removed'] = removed_line_count
        updated_record['duplicate_pages_removed'] = duplicate_pages_removed
        cleaned_records.append(updated_record)

        if repeated_lines or duplicate_pages_removed:
            drop_logs['page_or_section_deduplication'].append({
                'source_pdf': record['source_pdf'],
                'repeated_line_patterns_removed': len(repeated_lines),
                'repeated_line_instances_removed': removed_line_count,
                'duplicate_pages_removed': duplicate_pages_removed,
            })

    return cleaned_records


deduplicated_page_records = deduplicate_pages_and_repeated_sections(english_records)

add_stage(
    stage='3. After page/section deduplication',
    records=deduplicated_page_records,
    note='Removed repeated headers/footers and exact duplicate page hashes',
)

total_duplicate_pages = sum(record.get('duplicate_pages_removed', 0) for record in deduplicated_page_records)
total_repeated_line_instances = sum(record.get('repeated_line_instances_removed', 0) for record in deduplicated_page_records)

print(f'Documents kept after page/section deduplication: {len(deduplicated_page_records)}')
print(f'Exact duplicate pages removed: {total_duplicate_pages}')
print(f'Repeated header/footer line instances removed: {total_repeated_line_instances}')

Documents kept after page/section deduplication: 10
Exact duplicate pages removed: 0
Repeated header/footer line instances removed: 0


## 4. Length filter

After removing repeated content, we remove documents shorter than `5,000` characters. We do the length check after page/section deduplication so duplicated boilerplate does not artificially make a weak document look useful.

In [24]:
length_filtered_records = []

for record in deduplicated_page_records:
    character_count = len(record['text'])

    if character_count >= MIN_DOCUMENT_CHARACTERS:
        length_filtered_records.append(record)
    else:
        drop_logs['length_filter'].append({
            'source_pdf': record['source_pdf'],
            'characters': character_count,
            'minimum_required_characters': MIN_DOCUMENT_CHARACTERS,
            'reason': 'Document was too short after cleanup to be useful for instruction generation.',
        })

add_stage(
    stage='4. After length filter',
    records=length_filtered_records,
    note=f'Kept documents with >= {MIN_DOCUMENT_CHARACTERS} characters',
)

print(f'Documents kept after length filtering: {len(length_filtered_records)}')
print(f'Documents removed by length filtering: {len(drop_logs["length_filter"])}')

Documents kept after length filtering: 10
Documents removed by length filtering: 0


## 5. Near-duplicate document filter

Hashing catches exact duplicate pages, but it may miss documents that are almost the same with small formatting differences. For near-duplicate documents, we use **word shingles**:

- Convert each document into overlapping 5-word chunks.
- Compare two documents using Jaccard similarity.
- If similarity is at least `0.85`, keep the longer document and drop the shorter one.

In [25]:
def make_word_shingles(text, shingle_size=5):
    """Represent a document as a set of overlapping word chunks."""
    normalized = normalize_for_duplicate_detection(text)
    words = normalized.split()

    if len(words) < shingle_size:
        return set(words)

    shingles = set()
    for index in range(len(words) - shingle_size + 1):
        shingles.add(' '.join(words[index:index + shingle_size]))

    return shingles


def jaccard_similarity(left, right):
    """Jaccard similarity = overlap size divided by union size."""
    if not left and not right:
        return 1.0
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


# Sort longest-first so that if two documents are near-duplicates, we keep the richer one.
records_by_length = sorted(length_filtered_records, key=lambda record: len(record['text']), reverse=True)
unique_records_with_shingles = []

for record in tqdm(records_by_length, desc='Checking near-duplicate documents'):
    record_shingles = make_word_shingles(record['text'], SHINGLE_SIZE)

    best_match_pdf = None
    best_match_score = 0.0

    for kept_record in unique_records_with_shingles:
        score = jaccard_similarity(record_shingles, kept_record['_shingles'])
        if score > best_match_score:
            best_match_score = score
            best_match_pdf = kept_record['source_pdf']

    if best_match_score >= DOC_DUPLICATE_JACCARD_THRESHOLD:
        drop_logs['document_near_duplicate_filter'].append({
            'source_pdf': record['source_pdf'],
            'matched_pdf': best_match_pdf,
            'jaccard_similarity': round(best_match_score, 4),
            'reason': 'Near-duplicate document; kept the longer matching document.',
        })
        continue

    updated_record = record.copy()
    updated_record['_shingles'] = record_shingles
    unique_records_with_shingles.append(updated_record)

# Remove temporary shingle sets before saving records to JSON/text.
final_records = []
for record in sorted(unique_records_with_shingles, key=lambda item: item['source_index']):
    cleaned_record = record.copy()
    cleaned_record.pop('_shingles', None)
    final_records.append(cleaned_record)

add_stage(
    stage='5. After near-duplicate document filter',
    records=final_records,
    note=f'Removed document pairs with Jaccard similarity >= {DOC_DUPLICATE_JACCARD_THRESHOLD}',
)

print(f'Final cleaned documents: {len(final_records)}')
print(f'Documents removed as near-duplicates: {len(drop_logs["document_near_duplicate_filter"])}')

if len(final_records) < 5:
    print('WARNING: fewer than 5 cleaned documents remain. Add more source PDFs before submitting the assignment.')

Checking near-duplicate documents: 100%|██████████| 10/10 [00:00<00:00, 469.86it/s]

Final cleaned documents: 10
Documents removed as near-duplicates: 0


## 6. Save final cleaned `.txt` files

The assignment requires each cleaned document to be saved as a separate `.txt` file inside `domain_corpus/`. We also save a JSON manifest so the cleaning choices, final files, and dropped-document reasons are reproducible.

In [26]:
manifest_path = OUTPUT_DIR / 'corpus_manifest.json'

# To avoid stale files from a previous run, delete only files that this notebook previously wrote.
# We get that list from the previous manifest instead of deleting everything in domain_corpus/.
if manifest_path.exists():
    try:
        previous_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        for file_info in previous_manifest.get('final_files', []):
            old_file = OUTPUT_DIR / file_info.get('txt_file', '')
            if old_file.exists() and old_file.is_file():
                old_file.unlink()
    except json.JSONDecodeError:
        print('Previous manifest could not be parsed, so no old generated files were removed.')

final_file_rows = []

for index, record in enumerate(final_records, start=1):
    stem = safe_stem(record['source_pdf'])
    txt_filename = f'{index:02d}_{stem}.txt'
    output_path = OUTPUT_DIR / txt_filename

    output_path.write_text(record['text'], encoding='utf-8')

    final_file_rows.append({
        'txt_file': txt_filename,
        'source_pdf': record['source_pdf'],
        'pages': len(record['pages']),
        'characters': len(record['text']),
        'words': count_words(record['text']),
        'detected_language': record.get('detected_language', 'unknown'),
        'language_confidence': record.get('language_confidence', 0.0),
    })

manifest = {
    'domain': DOMAIN,
    'model_id_for_later_parts': MODEL_ID,
    'cleaning_parameters': {
        'minimum_document_characters': MIN_DOCUMENT_CHARACTERS,
        'english_confidence_threshold': ENGLISH_CONFIDENCE_THRESHOLD,
        'document_duplicate_method': '5-word shingle Jaccard similarity',
        'document_duplicate_jaccard_threshold': DOC_DUPLICATE_JACCARD_THRESHOLD,
        'page_duplicate_method': 'SHA-256 hash of normalized page text',
        'language_detection_library': 'langdetect',
        'pdf_text_extraction_library': 'PyMuPDF / fitz',
    },
    'stage_stats': stage_stats,
    'final_files': final_file_rows,
    'drop_logs': drop_logs,
}

manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(WORK_DIR / 'corpus_cleaning_report.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print(f'Saved {len(final_file_rows)} cleaned .txt files in: {OUTPUT_DIR.resolve()}')
print(f'Saved cleaning manifest: {manifest_path.resolve()}')
print(f'Saved report copy: {(WORK_DIR / "corpus_cleaning_report.json").resolve()}')

Saved 10 cleaned .txt files in: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/domain_corpus
Saved cleaning manifest: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/domain_corpus/corpus_manifest.json
Saved report copy: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/part_a_work/corpus_cleaning_report.json


## 7. Required corpus statistics report

Run this cell after saving the corpus. The table gives the numbers needed in your assignment report, and the final paragraph identifies which step reduced the corpus the most.

In [27]:
report_columns = ['stage', 'source_documents', 'txt_files', 'pages', 'characters', 'words', 'note']
print_table(stage_stats, report_columns)

reductions = []
for before, after in zip(stage_stats, stage_stats[1:]):
    reductions.append({
        'transition': f'{before["stage"]} -> {after["stage"]}',
        'documents_removed': before['source_documents'] - after['source_documents'],
        'pages_removed': before['pages'] - after['pages'],
        'characters_removed': before['characters'] - after['characters'],
    })

print('\nCleaning reductions:')
print_table(reductions, ['transition', 'documents_removed', 'pages_removed', 'characters_removed'])

positive_reductions = [
    row for row in reductions
    if row['documents_removed'] > 0 or row['pages_removed'] > 0 or row['characters_removed'] > 0
]

def explain_biggest_reduction(transition):
    transition_lower = transition.lower()
    if 'language' in transition_lower:
        return 'This step removes PDFs whose extracted text is not confidently English according to langdetect.'
    if 'page/section' in transition_lower:
        return 'This step removes repeated boilerplate, repeated headers/footers, and exact duplicate pages found by hashing normalized page text.'
    if 'length' in transition_lower:
        return 'This step removes short PDFs such as abstracts, short handouts, or mostly blank/scanned files that do not contain enough useful training text.'
    if 'near-duplicate' in transition_lower:
        return 'This step removes documents that are substantially the same as another retained document according to word-shingle Jaccard similarity.'
    if 'text extraction' in transition_lower:
        return 'This step removes unreadable PDFs or PDFs with no extractable text, often because they are scanned images without OCR.'
    return 'This step caused the largest measurable reduction in the corpus.'

if positive_reductions:
    biggest = max(
        positive_reductions,
        key=lambda row: (row['documents_removed'], row['pages_removed'], row['characters_removed']),
    )
    print('\nStep that reduced the corpus the most:')
    print(biggest['transition'])
    print(explain_biggest_reduction(biggest['transition']))
else:
    print('\nNo cleaning step reduced the corpus. All collected PDFs passed the filters.')

print('\nFinal cleaned corpus files:')
print_table(final_file_rows, ['txt_file', 'source_pdf', 'pages', 'characters', 'words', 'detected_language', 'language_confidence'])

stage                                   | source_documents | txt_files | pages | characters | words | note                                                            
----------------------------------------+------------------+-----------+-------+------------+-------+-----------------------------------------------------------------
0. Raw PDFs collected                   | 10               | 0         | 37    | 0          | 0     | Original PDFs placed in raw_pdfs/                               
1. After text extraction                | 10               | 10        | 37    | 143219     | 22923 | Readable page text extracted with PyMuPDF                       
2. After English language filter        | 10               | 10        | 37    | 143219     | 22923 | Kept lang=en with confidence >= 0.85                            
3. After page/section deduplication     | 10               | 10        | 37    | 143219     | 22923 | Removed repeated headers/footers and exact duplicate page hashe

## 8. Evaluator-facing dataset preview

The previous cells prove that we cleaned the corpus. This section proves that the cleaned dataset is actually useful and understandable. It prints representative examples, topic coverage, and short excerpts from the final `.txt` files.

Run this only after the cleaning pipeline has created `domain_corpus/`.

In [28]:
# -------------------------------------------
# Evaluator-friendly corpus overview
# -------------------------------------------
# Why this cell exists:
# An evaluator should not have to manually open every .txt file to understand the dataset.
# This cell prints a compact overview showing the cleaned files, their source PDFs,
# document sizes, and the medical topic each file roughly covers.

from pathlib import Path
import json
import re
from collections import Counter

# If this cell is run independently, recreate the output path variable.
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path('domain_corpus')

# If print_table was not defined because only this section was run, define a small fallback.
try:
    print_table
except NameError:
    def print_table(rows, columns):
        if not rows:
            print('No rows to show.')
            return
        widths = {}
        for column in columns:
            widths[column] = max([len(column)] + [len(str(row.get(column, ''))) for row in rows])
        print(' | '.join(column.ljust(widths[column]) for column in columns))
        print('-+-'.join('-' * widths[column] for column in columns))
        for row in rows:
            print(' | '.join(str(row.get(column, '')).ljust(widths[column]) for column in columns))


def simple_word_count(text):
    """Count words without depending on earlier notebook cells."""
    return len(re.findall(r'\b\w+\b', text))


def infer_topic_from_name(file_name):
    """Convert a file name into a readable medical topic label."""
    name = file_name.lower()
    topic_map = {
        'paracetamol': 'Paracetamol: pain and fever relief',
        'ibuprofen': 'Ibuprofen: pain, fever, and inflammation',
        'amoxicillin': 'Amoxicillin: antibiotic use',
        'metformin': 'Metformin: type 2 diabetes medicine',
        'salbutamol': 'Salbutamol: asthma inhaler',
        'cetirizine': 'Cetirizine: allergy medicine',
        'loratadine': 'Loratadine: allergy medicine',
        'aspirin': 'Aspirin: clot prevention / heart-stroke risk',
        'loperamide': 'Loperamide: diarrhoea treatment',
        'ferrous': 'Ferrous sulfate: iron-deficiency anaemia',
    }
    for keyword, topic in topic_map.items():
        if keyword in name:
            return topic
    return 'General patient medicine information'


corpus_files = sorted(OUTPUT_DIR.glob('*.txt'))
if not corpus_files:
    raise FileNotFoundError(
        f'No cleaned .txt files found in {OUTPUT_DIR.resolve()}. '
        'Run the Part A cleaning pipeline first so domain_corpus/ is generated.'
    )

# The manifest contains useful metadata such as source PDF, pages, characters, and words.
# If the manifest is missing, we still compute the important values directly from text files.
manifest_path = OUTPUT_DIR / 'corpus_manifest.json'
manifest = {}
manifest_lookup = {}
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    manifest_lookup = {
        row['txt_file']: row
        for row in manifest.get('final_files', [])
    }

overview_rows = []
total_characters = 0
total_words = 0

for path in corpus_files:
    text = path.read_text(encoding='utf-8')
    metadata = manifest_lookup.get(path.name, {})
    characters = metadata.get('characters', len(text))
    words = metadata.get('words', simple_word_count(text))
    total_characters += characters
    total_words += words

    overview_rows.append({
        'cleaned_file': path.name,
        'topic': infer_topic_from_name(path.name),
        'source_pdf': metadata.get('source_pdf', 'not recorded'),
        'pages': metadata.get('pages', 'n/a'),
        'words': words,
        'characters': characters,
    })

print('CLEANED DATASET OVERVIEW')
print(f'Cleaned text files: {len(corpus_files)}')
print(f'Total words: {total_words:,}')
print(f'Total characters: {total_characters:,}')
print(f'Average words per file: {round(total_words / len(corpus_files)):,}')
print('\nWhy this is a good beginner dataset:')
print('- It uses patient-facing medical leaflets, so the language is easier than research papers.')
print('- It covers common medicines and common patient questions: uses, dosage, warnings, side effects, and storage.')
print('- Each cleaned .txt file still maps back to a named source PDF for traceability.')
print('\nCleaned files and topics:')
print_table(overview_rows, ['cleaned_file', 'topic', 'source_pdf', 'pages', 'words', 'characters'])

CLEANED DATASET OVERVIEW
Cleaned text files: 10
Total words: 22,923
Total characters: 143,219
Average words per file: 2,292

Why this is a good beginner dataset:
- It uses patient-facing medical leaflets, so the language is easier than research papers.
- It covers common medicines and common patient questions: uses, dosage, warnings, side effects, and storage.
- Each cleaned .txt file still maps back to a named source PDF for traceability.

Cleaned files and topics:
cleaned_file                            | topic                                        | source_pdf                           | pages | words | characters
----------------------------------------+----------------------------------------------+--------------------------------------+-------+-------+-----------
01_01_paracetamol_500mg_tablets.txt     | Paracetamol: pain and fever relief           | 01_paracetamol_500mg_tablets.pdf     | 1     | 1441  | 8949      
02_02_ibuprofen_400mg_tablets.txt       | Ibuprofen: pain, fever

In [29]:
# -------------------------------------------
# Topic and keyword coverage
# -------------------------------------------
# This cell prints evidence that the dataset is not random medical text.
# It checks for medically useful categories that will later help instruction tuning:
# medicine names, dosing language, safety warnings, side effects, and patient groups.

all_corpus_text = '\n\n'.join(path.read_text(encoding='utf-8').lower() for path in corpus_files)

keyword_groups = {
    'Medicine names': [
        'paracetamol', 'ibuprofen', 'amoxicillin', 'metformin', 'salbutamol',
        'cetirizine', 'loratadine', 'aspirin', 'loperamide', 'ferrous sulfate'
    ],
    'Use cases / conditions': [
        'pain', 'fever', 'infection', 'diabetes', 'asthma', 'allergy',
        'diarrhoea', 'anaemia', 'heart attack', 'stroke'
    ],
    'Dosage and administration': [
        'dose', 'take', 'tablet', 'capsule', 'inhaler', 'swallow',
        'once a day', 'twice daily', '24 hours'
    ],
    'Safety and warnings': [
        'do not take', 'warnings', 'allergic', 'side effects', 'overdose',
        'pregnancy', 'breast-feeding', 'children', 'doctor', 'pharmacist'
    ],
}

coverage_rows = []
for group_name, keywords in keyword_groups.items():
    found_keywords = []
    total_mentions = 0

    for keyword in keywords:
        mentions = all_corpus_text.count(keyword)
        if mentions > 0:
            found_keywords.append(keyword)
            total_mentions += mentions

    coverage_rows.append({
        'coverage_area': group_name,
        'keywords_found': ', '.join(found_keywords[:8]) + (' ...' if len(found_keywords) > 8 else ''),
        'matched_keywords': f'{len(found_keywords)} / {len(keywords)}',
        'total_mentions': total_mentions,
    })

print('MEDICAL TOPIC COVERAGE CHECK')
print_table(coverage_rows, ['coverage_area', 'matched_keywords', 'total_mentions', 'keywords_found'])

print('\nInterpretation for evaluator:')
print('The corpus contains medicine names plus practical patient-care language such as dose, warnings, side effects, overdose, pregnancy, children, doctor, and pharmacist.')

MEDICAL TOPIC COVERAGE CHECK
coverage_area             | matched_keywords | total_mentions | keywords_found                                                                                  
--------------------------+------------------+----------------+-------------------------------------------------------------------------------------------------
Medicine names            | 10 / 10          | 453            | paracetamol, ibuprofen, amoxicillin, metformin, salbutamol, cetirizine, loratadine, aspirin ... 
Use cases / conditions    | 10 / 10          | 238            | pain, fever, infection, diabetes, asthma, allergy, diarrhoea, anaemia ...                       
Dosage and administration | 9 / 9            | 832            | dose, take, tablet, capsule, inhaler, swallow, once a day, twice daily ...                      
Safety and warnings       | 10 / 10          | 730            | do not take, warnings, allergic, side effects, overdose, pregnancy, breast-feeding, children ...

Inte

In [30]:
# -------------------------------------------
# Print representative examples from the cleaned dataset
# -------------------------------------------
# We print short excerpts, not full documents, because the evaluator only needs to see
# that the text is readable, relevant, and clean. Later cells can still use the full .txt files.

def compact_excerpt(text, max_characters=650):
    """Create a readable one-paragraph excerpt from cleaned text."""
    # Remove our page markers from the preview only. The actual saved corpus still keeps them.
    text = re.sub(r'\[Source page \d+\]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if len(text) <= max_characters:
        return text

    # Cut at a word boundary so the preview does not end in the middle of a word.
    shortened = text[:max_characters].rsplit(' ', 1)[0]
    return shortened + ' ...'


def choose_showcase_files(paths):
    """Choose a diverse set of files to display first."""
    preferred_topics = ['paracetamol', 'ibuprofen', 'amoxicillin', 'metformin', 'salbutamol']
    selected = []

    for topic in preferred_topics:
        match = next((path for path in paths if topic in path.name.lower()), None)
        if match and match not in selected:
            selected.append(match)

    # If any preferred topic did not survive cleaning, fill the list with other corpus files.
    for path in paths:
        if len(selected) >= 5:
            break
        if path not in selected:
            selected.append(path)

    return selected


showcase_files = choose_showcase_files(corpus_files)

print('REPRESENTATIVE CLEANED DATASET EXAMPLES')
for example_number, path in enumerate(showcase_files, start=1):
    text = path.read_text(encoding='utf-8')
    metadata = manifest_lookup.get(path.name, {})

    print('\n' + '=' * 90)
    print(f'Example {example_number}: {path.name}')
    print(f'Topic: {infer_topic_from_name(path.name)}')
    print(f'Source PDF: {metadata.get("source_pdf", "not recorded")}')
    print(f'Words: {metadata.get("words", simple_word_count(text)):,}')
    print('-' * 90)
    print(compact_excerpt(text))

print('\n' + '=' * 90)
print('These examples show the evaluator that the final corpus is readable, medically relevant, and suitable for creating instruction-answer pairs in Part B.')

REPRESENTATIVE CLEANED DATASET EXAMPLES

Example 1: 01_01_paracetamol_500mg_tablets.txt
Topic: Paracetamol: pain and fever relief
Source PDF: 01_paracetamol_500mg_tablets.pdf
Words: 1,441
------------------------------------------------------------------------------------------
Dosage: Adults, the elderly and children 16 years or older Take special care before taking Paracetamol Tablets: • if you have severe kidney or liver problems including non-cirrhotic alcoholic liver disease • if you have non-serious arthritis and need to take painkillers everyday 165 mm 165 mm 255 mm Package Leaflet: Information for the User Read all of this leaflet carefully before you start taking this medicine because it contains important information for you. Always take this medicine exactly as described in this leaflet or as your doctor or pharmacist has told you. • Keep this leaflet. You may need to read it again. • Ask your pharmacist ...

Example 2: 02_02_ibuprofen_400mg_tablets.txt
Topic: Ibuprofen: pai

In [31]:
# -------------------------------------------
# Save a small evaluator preview file
# -------------------------------------------
# Notebook output is useful, but a separate Markdown file is easy to attach or open.
# This file is only a preview; the actual dataset remains the .txt files in domain_corpus/.

preview_path = OUTPUT_DIR / 'EVALUATOR_DATASET_PREVIEW.md'
preview_lines = []

preview_lines.append('# Evaluator Dataset Preview')
preview_lines.append('')
preview_lines.append(f'- Domain: Medical & Clinical Literature')
preview_lines.append(f'- Cleaned text files: {len(corpus_files)}')
preview_lines.append(f'- Total words: {total_words:,}')
preview_lines.append(f'- Total characters: {total_characters:,}')
preview_lines.append('- Source type: patient-facing medicine information leaflets')
preview_lines.append('')

preview_lines.append('## Cleaned Files')
preview_lines.append('')
for row in overview_rows:
    preview_lines.append(
        f'- `{row["cleaned_file"]}`: {row["topic"]}; '
        f'{row["words"]:,} words; source `{row["source_pdf"]}`'
    )

preview_lines.append('')
preview_lines.append('## Representative Excerpts')
preview_lines.append('')

for example_number, path in enumerate(showcase_files, start=1):
    text = path.read_text(encoding='utf-8')
    preview_lines.append(f'### Example {example_number}: `{path.name}`')
    preview_lines.append('')
    preview_lines.append(f'**Topic:** {infer_topic_from_name(path.name)}')
    preview_lines.append('')
    preview_lines.append(compact_excerpt(text, max_characters=900))
    preview_lines.append('')

preview_path.write_text('\n'.join(preview_lines), encoding='utf-8')

print(f'Saved evaluator preview file: {preview_path.resolve()}')

Saved evaluator preview file: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/domain_corpus/EVALUATOR_DATASET_PREVIEW.md


# Part B Step 2 - Baseline Model Output

## BioGPT-Large Baseline on Apple Silicon

**Model for this step:** `microsoft/biogpt-large`  
**Hardware target:** Apple Silicon M1 Pro MacBook  
**Precision requirement:** BF16 / `torch.bfloat16`

This section creates the **pre-adaptation baseline**. We load the base BioGPT-Large model, inspect its architecture, run exactly three biomedical prompts with **no system prompt**, and save the outputs to `baseline_outputs.csv`.

This is intentionally **not** a fine-tuning section. It does not use QLoRA, PEFT, quantization, speculative decoding, instruction dataset creation, or any Part C optimization code.

## Why We Need a Baseline

Before fine-tuning, we need to know what the original base model already says. Later, after adapter fine-tuning, we compare the fine-tuned answers against these baseline answers.

A good baseline section should show three things:

1. **Storage footprint:** where the model was downloaded and how large it is.
2. **Architecture:** what transformer components exist inside the model.
3. **Raw behavior:** what the unadapted model generates for biomedical prompts.

In [32]:
# ------------------------------------------------------------
# Install dependencies for this self-contained section
# ------------------------------------------------------------
# torch: model loading and tensor computation
# transformers: AutoTokenizer and AutoModelForCausalLM
# accelerate: memory-efficient model loading through Hugging Face
# huggingface_hub: optional authentication through the HF_TOKEN environment variable
# pandas: clean tables and CSV saving
# safetensors: common modern model-weight format

%pip install -q torch transformers accelerate huggingface_hub pandas safetensors sacremoses

Note: you may need to restart the kernel to use updated packages.


## Optional Hugging Face Authentication

Hugging Face allows many public models to be downloaded anonymously, so this notebook remains runnable even if no token is provided.

Authenticated downloads are still useful because they usually have higher rate limits. Some gated or restricted models also require authentication before `from_pretrained(...)` can download them.

This notebook reads the token only from the environment variable `HF_TOKEN`. We do **not** hardcode tokens in the notebook.

In [ ]:
# ------------------------------------------------------------
# Optional Hugging Face authentication
# ------------------------------------------------------------
# Never paste a token directly into a notebook. Put it in the HF_TOKEN
# environment variable before launching Jupyter/Colab if authentication is needed.

import os
from huggingface_hub import login

#hf_token = os.environ.get('HF_TOKEN')
hf_token = "hf_"  # For testing only. Remove this line and set HF_TOKEN in the environment for real use.

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face authentication successful using HF_TOKEN.')
else:
    print('HF_TOKEN not found. Proceeding with anonymous access.')

Hugging Face authentication successful using HF_TOKEN.


In [34]:
# ------------------------------------------------------------
# Imports and reproducibility setup
# ------------------------------------------------------------

from pathlib import Path
from collections import Counter
import gc
import os
import platform
import re

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# A fixed seed makes deterministic decoding and any internal random choices reproducible.
set_seed(42)

print(f'PyTorch version: {torch.__version__}')
print(f'MPS available on this Mac: {torch.backends.mps.is_available()}')
print(f'CUDA available: {torch.cuda.is_available()}')

PyTorch version: 2.12.0
MPS available on this Mac: True
CUDA available: False


## Configuration

The model is loaded directly from Hugging Face using `AutoTokenizer.from_pretrained()` and `AutoModelForCausalLM.from_pretrained()`.

After loading, we save the model and tokenizer locally to `./local_models/BioGPT` using `save_pretrained(...)`. On Apple Silicon, the notebook detects MPS and uses it when available.

In [35]:
# ------------------------------------------------------------
# User-configurable settings
# ------------------------------------------------------------

MODEL_ID = 'microsoft/biogpt-large'

# Local folder required by the assignment after loading the model.
LOCAL_SAVE_DIR = Path('local_models') / 'BioGPT'

# Baseline output file required by the assignment.
BASELINE_CSV_PATH = Path('baseline_outputs.csv')

# BF16 stores each parameter in 2 bytes.
BF16_DTYPE = torch.bfloat16
BF16_BYTES_PER_PARAMETER = 2

# Keep generation short enough for a laptop baseline run but long enough to inspect answer quality.
MAX_NEW_TOKENS = 96

print(f'Model ID: {MODEL_ID}')
print(f'Local save folder: {LOCAL_SAVE_DIR.resolve()}')
print(f'Baseline CSV will be saved to: {BASELINE_CSV_PATH.resolve()}')
print(f'Requested model dtype: {BF16_DTYPE}')

Model ID: microsoft/biogpt-large
Local save folder: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/local_models/BioGPT
Baseline CSV will be saved to: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/baseline_outputs.csv
Requested model dtype: torch.bfloat16


## Storage Helper Functions

This notebook uses the standard Hugging Face loading workflow. After the model is loaded, we save it locally using:

```python
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)
```

The helper functions below are used later to calculate total local folder size, largest files, and storage estimates.

In [36]:
# ------------------------------------------------------------
# Storage reporting helpers
# ------------------------------------------------------------
# There is no separate low-level Hub download step in this workflow.
# Transformers downloads the model internally during from_pretrained(...).
# After loading, we save the model ourselves into LOCAL_SAVE_DIR and measure that folder.


def bytes_to_gb(num_bytes):
    """Convert bytes to GiB, the usual unit used for model storage and memory."""
    return num_bytes / (1024 ** 3)


def bytes_to_mb(num_bytes):
    """Convert bytes to MiB for individual file reporting."""
    return num_bytes / (1024 ** 2)


def folder_size_bytes(folder):
    """Return the recursive size of all files inside a folder."""
    return sum(path.stat().st_size for path in folder.rglob('*') if path.is_file())


def collect_major_model_files(folder):
    """Collect important model, tokenizer, and config files with file sizes."""
    major_suffixes = {'.bin', '.safetensors', '.json', '.model', '.txt', '.py'}
    important_exact_names = {
        'config.json',
        'generation_config.json',
        'tokenizer.json',
        'tokenizer_config.json',
        'special_tokens_map.json',
        'vocab.json',
        'merges.txt',
        'pytorch_model.bin',
        'model.safetensors',
    }

    rows = []
    for path in sorted(folder.rglob('*')):
        if not path.is_file():
            continue

        size_bytes = path.stat().st_size
        is_major = (
            path.name in important_exact_names
            or path.suffix in major_suffixes
            or size_bytes >= 1_000_000
        )

        if is_major:
            rows.append({
                'file': str(path.relative_to(folder)),
                'size_mb': round(bytes_to_mb(size_bytes), 2),
                'size_gb': round(bytes_to_gb(size_bytes), 4),
            })

    return pd.DataFrame(rows).sort_values('size_mb', ascending=False).reset_index(drop=True)


print('Storage helper functions are ready.')

Storage helper functions are ready.


## Load Tokenizer and BF16 Base Model

The tokenizer converts text into token IDs from the model's vocabulary. The model only understands these IDs, so the tokenizer must match the model.

We load BioGPT-Large with:

```python
AutoTokenizer.from_pretrained(MODEL_ID)
AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
```

Because this is a baseline, the model is put into `eval()` mode. That disables training-time behavior such as dropout.

In [37]:
# ------------------------------------------------------------
# Load tokenizer and base model in BF16
# ------------------------------------------------------------

# Always use the tokenizer paired with the selected model.
# This loads directly from Hugging Face using the standard transformers workflow.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# GPT-style causal language models often do not define a separate pad token.
# For generation, using EOS as PAD is a standard safe choice.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Device selection:
# 1. Use Apple Silicon MPS when available on M1/M2/M3 Macs.
# 2. Otherwise use CUDA when available.
# 3. Fall back to CPU.
is_apple_silicon = platform.system() == 'Darwin' and platform.machine() == 'arm64'
if is_apple_silicon and torch.backends.mps.is_available():
    device = torch.device('mps')
    device_reason = 'Apple Silicon detected and MPS is available.'
elif torch.cuda.is_available():
    device = torch.device('cuda')
    device_reason = 'CUDA GPU is available.'
else:
    device = torch.device('cpu')
    device_reason = 'No MPS/CUDA accelerator available; using CPU.'

print(f'Selected device: {device}')
print(device_reason)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=BF16_DTYPE,
    low_cpu_mem_usage=True,
)

# Save the loaded model and tokenizer locally as required by the assignment.
# We save before moving to MPS/CUDA so serialization happens from standard CPU tensors.
LOCAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)

total_folder_size = folder_size_bytes(LOCAL_SAVE_DIR)
major_files_df = collect_major_model_files(LOCAL_SAVE_DIR)

loaded_total_params = sum(parameter.numel() for parameter in model.parameters())
model_storage_estimate_gb = bytes_to_gb(loaded_total_params * BF16_BYTES_PER_PARAMETER)

# Move the model to the selected runtime device after local saving.
model.to(device)
model.eval()

# Confirm the loaded parameter dtypes.
dtype_counts = Counter(str(parameter.dtype) for parameter in model.parameters())
print('Parameter dtype counts:')
for dtype_name, count in dtype_counts.items():
    print(f'  {dtype_name}: {count:,} tensors')

print(f'Tokenizer vocabulary size: {len(tokenizer):,}')

print('\nLOAD SUMMARY')
print(f'Model name: {MODEL_ID}')
print(f'Device used: {device}')
print(f'Dtype used: {BF16_DTYPE}')
print(f'Local saved model path: {LOCAL_SAVE_DIR.resolve()}')
print(f'Total local folder size: {bytes_to_gb(total_folder_size):.3f} GB')

print('\nLargest local model/tokenizer/config files:')
display(major_files_df.head(10))

print('\nModel storage estimate:')
print(f'Total parameters counted after load: {loaded_total_params:,}')
print(f'BF16 uses {BF16_BYTES_PER_PARAMETER} bytes per parameter.')
print(f'Estimated BF16 parameter storage: {model_storage_estimate_gb:.3f} GB')

Selected device: mps
Apple Silicon detected and MPS is available.


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.03s/it]


Parameter dtype counts:
  torch.bfloat16: 772 tensors
Tokenizer vocabulary size: 57,717

LOAD SUMMARY
Model name: microsoft/biogpt-large
Device used: mps
Dtype used: torch.bfloat16
Local saved model path: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/local_models/BioGPT
Total local folder size: 2.928 GB

Largest local model/tokenizer/config files:


,file,size_mb,size_gb
0,model.safetensors,2996.89,2.9266
1,vocab.json,1.18,0.0012
2,merges.txt,0.54,0.0005
3,config.json,0.00,0.0000
4,generation_config.json,0.00,0.0000
5,tokenizer_config.json,0.00,0.0000



Model storage estimate:
Total parameters counted after load: 1,571,188,800
BF16 uses 2 bytes per parameter.
Estimated BF16 parameter storage: 2.927 GB


## Transformer Architecture Theory Map

BioGPT-Large is a causal decoder-only language model. At a high level, a decoder-only transformer contains:

- **Embedding layer:** converts token IDs into dense vectors.
- **Decoder stack:** repeats many transformer decoder layers. Each layer contains self-attention, a feed-forward network, and normalization.
- **Query/Key/Value projections:** create the tensors used by self-attention to decide which previous tokens matter.
- **Attention output projection:** mixes attention-head outputs back into the model's hidden space.
- **Feed-forward network:** applies per-token nonlinear transformations after attention.
- **Final normalization:** stabilizes the final hidden states before prediction.
- **LM head / unembedding layer:** converts hidden states back into vocabulary logits so the model can choose the next token.

In [38]:
# ------------------------------------------------------------
# Generic architecture-inspection helper functions
# ------------------------------------------------------------
# These helpers support GPT-style models such as BioGPT-Large and are written defensively
# so the notebook prints useful information even if module names differ slightly.


def count_parameters(module):
    """Count parameters in a model or submodule."""
    return sum(parameter.numel() for parameter in module.parameters())


def count_trainable_parameters(module):
    """Count parameters that would be updated if we trained the model."""
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


def get_config_value(config, names, default=None):
    """Return the first available config attribute from a list of possible names."""
    for name in names:
        if hasattr(config, name):
            value = getattr(config, name)
            if value is not None:
                return value
    return default


def get_nested_module(root, dotted_path):
    """Fetch a nested module by a dotted path such as 'transformer.h'."""
    current = root
    for part in dotted_path.split('.'):
        if part.isdigit():
            current = current[int(part)]
        else:
            current = getattr(current, part)
    return current


def find_first_existing_module(root, candidate_paths):
    """Try multiple common paths and return the first module that exists."""
    for path in candidate_paths:
        try:
            return path, get_nested_module(root, path)
        except Exception:
            continue
    return None, None


def find_decoder_layers(root):
    """Find the ModuleList containing decoder layers."""
    candidate_paths = [
        'biogpt.layers',
        'transformer.h',
        'model.layers',
        'model.decoder.layers',
        'decoder.layers',
        'gpt_neox.layers',
    ]

    path, layers = find_first_existing_module(root, candidate_paths)
    if layers is None:
        raise ValueError('Could not locate decoder layers. Print the model structure and add the path manually.')

    return path, layers


def module_shape(module):
    """Return the main weight shape for a module, if it has one."""
    if module is None:
        return 'not found'
    if hasattr(module, 'weight') and module.weight is not None:
        return tuple(module.weight.shape)
    return 'no direct weight tensor'


def find_attention_block(layer):
    """Find the self-attention block inside one decoder layer."""
    for name in ['attn', 'self_attn', 'attention']:
        if hasattr(layer, name):
            return name, getattr(layer, name)

    for name, module in layer.named_modules():
        lowered = name.lower()
        if 'attn' in lowered or 'attention' in lowered:
            return name, module

    return None, None


def find_feed_forward_block(layer):
    """Find the MLP/feed-forward block inside one decoder layer."""
    for name in ['mlp', 'feed_forward', 'ffn']:
        if hasattr(layer, name):
            return name, getattr(layer, name)

    # BioGPT decoder layers expose the feed-forward projections directly as fc1 and fc2.
    if hasattr(layer, 'fc1') and hasattr(layer, 'fc2'):
        return '(decoder layer direct fc1/fc2)', layer

    for name, module in layer.named_modules():
        lowered = name.lower()
        if 'mlp' in lowered or 'ffn' in lowered or 'feed_forward' in lowered:
            return name, module

    return None, None


def find_layer_norms(layer):
    """Find normalization modules inside one decoder layer."""
    norm_rows = []
    for name, module in layer.named_modules():
        module_class = module.__class__.__name__.lower()
        if isinstance(module, torch.nn.LayerNorm) or 'norm' in name.lower() or 'layernorm' in module_class:
            norm_rows.append((name or '(layer itself)', module))
    return norm_rows


decoder_layer_path, decoder_layers = find_decoder_layers(model)
input_embeddings = model.get_input_embeddings()
lm_head = model.get_output_embeddings()
final_norm_path, final_norm = find_first_existing_module(
    model,
    [
        'biogpt.layer_norm',
        'transformer.ln_f',
        'model.norm',
        'model.decoder.final_layer_norm',
        'gpt_neox.final_layer_norm',
        'ln_f',
    ],
)

print(f'Decoder layer path: {decoder_layer_path}')
print(f'Number of decoder layers found: {len(decoder_layers)}')
print(f'Input embedding module: {input_embeddings.__class__.__name__}')
print(f'Final normalization path: {final_norm_path}')
print(f'LM head module: {lm_head.__class__.__name__ if lm_head is not None else "not found"}')

Decoder layer path: biogpt.layers
Number of decoder layers found: 48
Input embedding module: BioGptScaledWordEmbedding
Final normalization path: biogpt.layer_norm
LM head module: Linear


## Architecture Report

This report gives the evaluator the important model dimensions:

- **Total parameter count:** all learned weights in the model.
- **Trainable parameter count:** parameters with `requires_grad=True`. We are not training here, but this tells us the trainable capacity of the base model.
- **Decoder layers:** how many transformer blocks are stacked.
- **Hidden size:** the width of each token vector inside the model.
- **Vocabulary size:** how many possible tokens the model can predict.
- **Attention heads:** how many parallel attention views each layer uses.
- **Feed-forward dimension:** the inner width of the MLP block.
- **BF16 memory estimate:** approximate memory needed just to store parameters in BF16.

In [39]:
# ------------------------------------------------------------
# Architecture report
# ------------------------------------------------------------

config = model.config

total_params = count_parameters(model)
trainable_params = count_trainable_parameters(model)
num_decoder_layers = len(decoder_layers)
hidden_size = get_config_value(config, ['hidden_size', 'n_embd', 'd_model'])
vocab_size = get_config_value(config, ['vocab_size'])
num_attention_heads = get_config_value(config, ['num_attention_heads', 'n_head', 'num_heads'])
intermediate_size = get_config_value(config, ['intermediate_size', 'n_inner', 'ffn_dim'])

# GPT-2 style configs sometimes store n_inner as None. The standard MLP expansion is 4x hidden size.
if intermediate_size is None and hidden_size is not None:
    intermediate_size = 4 * hidden_size

bf16_parameter_memory_gb = bytes_to_gb(total_params * BF16_BYTES_PER_PARAMETER)

architecture_report = pd.DataFrame([
    {'Metric': 'model_name', 'Value': MODEL_ID},
    {'Metric': 'total_parameter_count', 'Value': f'{total_params:,}'},
    {'Metric': 'trainable_parameter_count', 'Value': f'{trainable_params:,}'},
    {'Metric': 'number_of_decoder_layers', 'Value': f'{num_decoder_layers:,}'},
    {'Metric': 'hidden_size', 'Value': f'{hidden_size:,}'},
    {'Metric': 'vocabulary_size', 'Value': f'{vocab_size:,}'},
    {'Metric': 'number_of_attention_heads', 'Value': f'{num_attention_heads:,}'},
    {'Metric': 'intermediate_feed_forward_dimension', 'Value': f'{intermediate_size:,}'},
    {'Metric': 'parameter_memory_estimate_bf16_gb', 'Value': f'{bf16_parameter_memory_gb:.3f} GB'},
])

print('ARCHITECTURE REPORT')
display(architecture_report)

ARCHITECTURE REPORT


,Metric,Value
0,model_name,microsoft/biogpt-large
1,total_parameter_count,"1,571,188,800"
2,trainable_parameter_count,"1,571,188,800"
3,number_of_decoder_layers,48
4,hidden_size,"1,600"
5,vocabulary_size,"57,717"
6,number_of_attention_heads,25
7,intermediate_feed_forward_dimension,"6,400"
8,parameter_memory_estimate_bf16_gb,2.927 GB


## High-Level Architecture Tree

The tree below maps the model into the standard decoder-only transformer structure.

Each **Decoder Layer** repeats the same pattern: normalization, self-attention, another normalization, and a feed-forward network. Stacking many layers allows the model to build increasingly abstract representations of the input text.

In [40]:
# ------------------------------------------------------------
# High-level architecture tree
# ------------------------------------------------------------

print(f'{MODEL_ID} - Causal Language Model')
print(f'|-- Embedding Layer: {input_embeddings.__class__.__name__}, weight shape={module_shape(input_embeddings)}')

for layer_index, layer in enumerate(decoder_layers, start=1):
    print(f'|-- Decoder Layer {layer_index}: {layer.__class__.__name__}')

print(f'|-- Final LayerNorm: {final_norm.__class__.__name__ if final_norm is not None else "not found"}')
print(f'`-- LM Head: {lm_head.__class__.__name__ if lm_head is not None else "not found"}, weight shape={module_shape(lm_head)}')

microsoft/biogpt-large - Causal Language Model
|-- Embedding Layer: BioGptScaledWordEmbedding, weight shape=(57717, 1600)
|-- Decoder Layer 1: BioGptDecoderLayer
|-- Decoder Layer 2: BioGptDecoderLayer
|-- Decoder Layer 3: BioGptDecoderLayer
|-- Decoder Layer 4: BioGptDecoderLayer
|-- Decoder Layer 5: BioGptDecoderLayer
|-- Decoder Layer 6: BioGptDecoderLayer
|-- Decoder Layer 7: BioGptDecoderLayer
|-- Decoder Layer 8: BioGptDecoderLayer
|-- Decoder Layer 9: BioGptDecoderLayer
|-- Decoder Layer 10: BioGptDecoderLayer
|-- Decoder Layer 11: BioGptDecoderLayer
|-- Decoder Layer 12: BioGptDecoderLayer
|-- Decoder Layer 13: BioGptDecoderLayer
|-- Decoder Layer 14: BioGptDecoderLayer
|-- Decoder Layer 15: BioGptDecoderLayer
|-- Decoder Layer 16: BioGptDecoderLayer
|-- Decoder Layer 17: BioGptDecoderLayer
|-- Decoder Layer 18: BioGptDecoderLayer
|-- Decoder Layer 19: BioGptDecoderLayer
|-- Decoder Layer 20: BioGptDecoderLayer
|-- Decoder Layer 21: BioGptDecoderLayer
|-- Decoder Layer 22: BioG

## Inspect Decoder Layer 1 in Detail

We inspect the first decoder layer because all decoder layers usually share the same structure. This shows the evaluator exactly where attention, feed-forward, and normalization live inside the model.

Transformer theory mapping:

- **Self-attention block:** decides which previous tokens each current token should attend to.
- **Q/K/V projections:** create query, key, and value vectors for attention.
- **Attention output projection:** mixes the attention result back into the hidden dimension.
- **Feed-forward block:** expands and compresses each token representation independently.
- **Layer norms:** stabilize activations and make deep transformer training/inference more reliable.

In [41]:
# ------------------------------------------------------------
# Full Decoder Layer 1 inspection
# ------------------------------------------------------------

layer1 = decoder_layers[0]
attention_name, attention_block = find_attention_block(layer1)
feed_forward_name, feed_forward_block = find_feed_forward_block(layer1)
layer_norms = find_layer_norms(layer1)

print('FULL MODULE STRUCTURE FOR DECODER LAYER 1')
print('-' * 80)
print(layer1)

print('\nALL SUBMODULES INSIDE DECODER LAYER 1')
print('-' * 80)
submodule_rows = []
for name, module in layer1.named_modules():
    submodule_rows.append({
        'submodule_name': name or '(layer itself)',
        'module_type': module.__class__.__name__,
        'parameter_count': count_parameters(module),
    })

display(pd.DataFrame(submodule_rows))

print('\nSELF-ATTENTION BLOCK')
print('-' * 80)
print(f'Attention module path inside layer 1: {attention_name}')
print(attention_block)

print('\nFEED-FORWARD / MLP BLOCK')
print('-' * 80)
print(f'Feed-forward module path inside layer 1: {feed_forward_name}')
print(feed_forward_block)

print('\nLAYER NORMS INSIDE LAYER 1')
print('-' * 80)
if layer_norms:
    for norm_name, norm_module in layer_norms:
        print(f'{norm_name}: {norm_module}')
else:
    print('No LayerNorm-like modules found by name/type inspection.')

FULL MODULE STRUCTURE FOR DECODER LAYER 1
--------------------------------------------------------------------------------
BioGptDecoderLayer(
  (self_attn): BioGptAttention(
    (k_proj): Linear(in_features=1600, out_features=1600, bias=True)
    (v_proj): Linear(in_features=1600, out_features=1600, bias=True)
    (q_proj): Linear(in_features=1600, out_features=1600, bias=True)
    (out_proj): Linear(in_features=1600, out_features=1600, bias=True)
  )
  (activation_fn): GELUActivation()
  (self_attn_layer_norm): LayerNorm((1600,), eps=1e-05, elementwise_affine=True, bias=True)
  (fc1): Linear(in_features=1600, out_features=6400, bias=True)
  (fc2): Linear(in_features=6400, out_features=1600, bias=True)
  (final_layer_norm): LayerNorm((1600,), eps=1e-05, elementwise_affine=True, bias=True)
)

ALL SUBMODULES INSIDE DECODER LAYER 1
--------------------------------------------------------------------------------


,submodule_name,module_type,parameter_count
0,(layer itself),BioGptDecoderLayer,30740800
1,self_attn,BioGptAttention,10246400
2,self_attn.k_proj,Linear,2561600
3,self_attn.v_proj,Linear,2561600
4,self_attn.q_proj,Linear,2561600
5,self_attn.out_proj,Linear,2561600
6,activation_fn,GELUActivation,0
7,self_attn_layer_norm,LayerNorm,3200
8,fc1,Linear,10246400
9,fc2,Linear,10241600



SELF-ATTENTION BLOCK
--------------------------------------------------------------------------------
Attention module path inside layer 1: self_attn
BioGptAttention(
  (k_proj): Linear(in_features=1600, out_features=1600, bias=True)
  (v_proj): Linear(in_features=1600, out_features=1600, bias=True)
  (q_proj): Linear(in_features=1600, out_features=1600, bias=True)
  (out_proj): Linear(in_features=1600, out_features=1600, bias=True)
)

FEED-FORWARD / MLP BLOCK
--------------------------------------------------------------------------------
Feed-forward module path inside layer 1: (decoder layer direct fc1/fc2)
BioGptDecoderLayer(
  (self_attn): BioGptAttention(
    (k_proj): Linear(in_features=1600, out_features=1600, bias=True)
    (v_proj): Linear(in_features=1600, out_features=1600, bias=True)
    (q_proj): Linear(in_features=1600, out_features=1600, bias=True)
    (out_proj): Linear(in_features=1600, out_features=1600, bias=True)
  )
  (activation_fn): GELUActivation()
  (self_att

## Weight Tensor Shapes for Layer 1

The next table prints the exact shapes requested in the assignment.

BioGPT normally exposes separate `q_proj`, `k_proj`, and `v_proj` modules inside the self-attention block. The notebook still contains a fallback for GPT-style models that store Q, K, and V together in one combined projection, because Q, K, and V are separate mathematical projections even when an implementation stores them together.

In [42]:
# ------------------------------------------------------------
# Projection and tensor-shape inspection
# ------------------------------------------------------------


def first_existing_attr(module, names):
    """Return the first child attribute that exists on a module."""
    if module is None:
        return None, None
    for name in names:
        if hasattr(module, name):
            return name, getattr(module, name)
    return None, None


def find_module_by_name_keywords(module, keywords):
    """Find a nested module whose name contains all requested keywords."""
    if module is None:
        return None, None
    for name, child in module.named_modules():
        lowered = name.lower()
        if all(keyword in lowered for keyword in keywords):
            return name, child
    return None, None


def parameter_count_from_shape(shape, has_bias=False, bias_size=None):
    """Compute parameter count from a weight shape and optional bias size."""
    if shape in [None, 'not found', 'no direct weight tensor']:
        return None
    total = 1
    for dimension in shape:
        total *= int(dimension)
    if has_bias:
        total += int(bias_size if bias_size is not None else shape[-1])
    return total


def module_parameter_count_or_none(module):
    """Count parameters for a module if it exists."""
    if module is None:
        return None
    return count_parameters(module)


def add_summary_row(rows, component, shape, parameter_count, source_module):
    """Append one clean row to the architecture summary table."""
    rows.append({
        'Component': component,
        'Shape': str(shape),
        'Parameter Count': 'not found' if parameter_count is None else f'{parameter_count:,}',
        'Source Module': source_module,
    })


summary_rows = []

# Embedding and LM head.
embedding_weight_shape = module_shape(input_embeddings)
lm_head_weight_shape = module_shape(lm_head)
embedding_param_count = module_parameter_count_or_none(input_embeddings)
lm_head_param_count = module_parameter_count_or_none(lm_head)

lm_head_tied = False
if input_embeddings is not None and lm_head is not None:
    try:
        lm_head_tied = input_embeddings.weight.data_ptr() == lm_head.weight.data_ptr()
    except Exception:
        lm_head_tied = False

add_summary_row(summary_rows, 'Embedding matrix', embedding_weight_shape, embedding_param_count, 'model.get_input_embeddings()')

# Attention projections. BioGPT exposes q_proj, k_proj, v_proj, and out_proj inside self_attn.
q_name, q_proj = first_existing_attr(attention_block, ['q_proj', 'query', 'q'])
k_name, k_proj = first_existing_attr(attention_block, ['k_proj', 'key', 'k'])
v_name, v_proj = first_existing_attr(attention_block, ['v_proj', 'value', 'v'])
out_name, out_proj = first_existing_attr(attention_block, ['out_proj', 'o_proj', 'c_proj', 'dense'])
combined_qkv_name, combined_qkv = first_existing_attr(attention_block, ['c_attn', 'query_key_value', 'qkv_proj'])

if q_proj is not None and k_proj is not None and v_proj is not None:
    add_summary_row(summary_rows, 'Layer 1 q_proj weight', module_shape(q_proj), count_parameters(q_proj), f'{attention_name}.{q_name}')
    add_summary_row(summary_rows, 'Layer 1 k_proj weight', module_shape(k_proj), count_parameters(k_proj), f'{attention_name}.{k_name}')
    add_summary_row(summary_rows, 'Layer 1 v_proj weight', module_shape(v_proj), count_parameters(v_proj), f'{attention_name}.{v_name}')
else:
    if combined_qkv is None:
        combined_qkv_name, combined_qkv = find_module_by_name_keywords(attention_block, ['attn'])

    combined_shape = module_shape(combined_qkv)
    has_combined_bias = hasattr(combined_qkv, 'bias') and combined_qkv.bias is not None

    # Conceptual split for GPT-style combined QKV projection.
    conceptual_qkv_shape = (hidden_size, hidden_size)
    conceptual_qkv_params = hidden_size * hidden_size + (hidden_size if has_combined_bias else 0)

    add_summary_row(summary_rows, 'Layer 1 q_proj weight (conceptual slice of combined QKV)', conceptual_qkv_shape, conceptual_qkv_params, f'{attention_name}.{combined_qkv_name}')
    add_summary_row(summary_rows, 'Layer 1 k_proj weight (conceptual slice of combined QKV)', conceptual_qkv_shape, conceptual_qkv_params, f'{attention_name}.{combined_qkv_name}')
    add_summary_row(summary_rows, 'Layer 1 v_proj weight (conceptual slice of combined QKV)', conceptual_qkv_shape, conceptual_qkv_params, f'{attention_name}.{combined_qkv_name}')
    print(f'Combined QKV module detected: {attention_name}.{combined_qkv_name}, actual weight shape={combined_shape}')

add_summary_row(summary_rows, 'Layer 1 attention out_proj weight', module_shape(out_proj), module_parameter_count_or_none(out_proj), f'{attention_name}.{out_name}')

# Feed-forward projections.
ff_in_name, ff_input_projection = first_existing_attr(feed_forward_block, ['fc1', 'c_fc', 'fc_in', 'up_proj', 'gate_proj', 'dense_h_to_4h'])
ff_out_name, ff_output_projection = first_existing_attr(feed_forward_block, ['fc2', 'c_proj', 'fc_out', 'down_proj', 'dense_4h_to_h'])

if ff_input_projection is None:
    ff_in_name, ff_input_projection = find_module_by_name_keywords(feed_forward_block, ['fc'])
if ff_output_projection is None:
    ff_out_name, ff_output_projection = find_module_by_name_keywords(feed_forward_block, ['proj'])

add_summary_row(summary_rows, 'Layer 1 feed-forward input projection', module_shape(ff_input_projection), module_parameter_count_or_none(ff_input_projection), f'{feed_forward_name}.{ff_in_name}')
add_summary_row(summary_rows, 'Layer 1 feed-forward output projection', module_shape(ff_output_projection), module_parameter_count_or_none(ff_output_projection), f'{feed_forward_name}.{ff_out_name}')

# Layer norms from layer 1.
for norm_name, norm_module in layer_norms:
    add_summary_row(summary_rows, f'Layer 1 norm: {norm_name}', module_shape(norm_module), count_parameters(norm_module), norm_name)

lm_head_component_name = 'LM head / unembedding matrix'
if lm_head_tied:
    lm_head_component_name += ' (tied to embedding weights)'
add_summary_row(summary_rows, lm_head_component_name, lm_head_weight_shape, lm_head_param_count, 'model.get_output_embeddings()')

architecture_summary_df = pd.DataFrame(summary_rows)
print('CLEAN ARCHITECTURE SUMMARY TABLE')
display(architecture_summary_df[['Component', 'Shape', 'Parameter Count']])

print('\nDetailed source-module mapping:')
display(architecture_summary_df)

CLEAN ARCHITECTURE SUMMARY TABLE


,Component,Shape,Parameter Count
0,Embedding matrix,"(57717, 1600)","92,347,200"
1,Layer 1 q_proj weight,"(1600, 1600)","2,561,600"
2,Layer 1 k_proj weight,"(1600, 1600)","2,561,600"
3,Layer 1 v_proj weight,"(1600, 1600)","2,561,600"
4,Layer 1 attention out_proj weight,"(1600, 1600)","2,561,600"
5,Layer 1 feed-forward input projection,"(6400, 1600)","10,246,400"
6,Layer 1 feed-forward output projection,"(1600, 6400)","10,241,600"
7,Layer 1 norm: self_attn_layer_norm,"(1600,)","3,200"
8,Layer 1 norm: final_layer_norm,"(1600,)","3,200"
9,LM head / unembedding matrix (tied to embeddin...,"(57717, 1600)","92,347,200"



Detailed source-module mapping:


,Component,Shape,Parameter Count,Source Module
0,Embedding matrix,"(57717, 1600)","92,347,200",model.get_input_embeddings()
1,Layer 1 q_proj weight,"(1600, 1600)","2,561,600",self_attn.q_proj
2,Layer 1 k_proj weight,"(1600, 1600)","2,561,600",self_attn.k_proj
3,Layer 1 v_proj weight,"(1600, 1600)","2,561,600",self_attn.v_proj
4,Layer 1 attention out_proj weight,"(1600, 1600)","2,561,600",self_attn.out_proj
5,Layer 1 feed-forward input projection,"(6400, 1600)","10,246,400",(decoder layer direct fc1/fc2).fc1
6,Layer 1 feed-forward output projection,"(1600, 6400)","10,241,600",(decoder layer direct fc1/fc2).fc2
7,Layer 1 norm: self_attn_layer_norm,"(1600,)","3,200",self_attn_layer_norm
8,Layer 1 norm: final_layer_norm,"(1600,)","3,200",final_layer_norm
9,LM head / unembedding matrix (tied to embeddin...,"(57717, 1600)","92,347,200",model.get_output_embeddings()


## Baseline Generation Prompts

The assignment requires exactly three biomedical prompts and no system prompt. That means we pass only the raw user prompt text into the tokenizer.

We use deterministic decoding with `do_sample=False` so the same model and prompt produce stable baseline outputs. These outputs are saved and reused later when comparing against the fine-tuned adapter.

In [43]:
# ------------------------------------------------------------
# Run exactly three baseline prompts, with no system prompt
# ------------------------------------------------------------

baseline_prompts = [
    'Insulin is a hormone produced by the pancreas',
    """
        Question: What are common symptoms of hypertension?
        Answer:
    """,
    'antibiotics are used to treat bacterial infections',
]

baseline_rows = []

for prompt_id, prompt in enumerate(baseline_prompts, start=1):
    print('\n' + '=' * 100)
    print(f'Prompt {prompt_id}: {prompt}')
    print('-' * 100)

    # No system prompt is added here. The raw prompt is tokenized directly.
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_token_count = inputs['input_ids'].shape[1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_token_ids = generated_ids[0, input_token_count:]
    generated_output = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()
    full_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

    print(generated_output)

    baseline_rows.append({
        'prompt_id': prompt_id,
        'model_id': MODEL_ID,
        'precision': 'bfloat16',
        'device': str(device),
        'system_prompt_used': False,
        'decoding_strategy': 'greedy decoding; do_sample=False; num_beams=1',
        'max_new_tokens': MAX_NEW_TOKENS,
        'prompt': prompt,
        'generated_output': generated_output,
        'full_prompt_plus_output': full_text,
    })

baseline_outputs_df = pd.DataFrame(baseline_rows)

print('\nBASELINE OUTPUTS DATAFRAME')
display(baseline_outputs_df)


Prompt 1: Insulin is a hormone produced by the pancreas
----------------------------------------------------------------------------------------------------
that regulates blood glucose levels. Insulin is secreted in response to elevated blood glucose levels and acts on target tissues to promote glucose uptake and storage. Insulin resistance is a condition in which cells in the body do not respond to normal levels of insulin. Insulin resistance is a major risk factor for type 2 diabetes. Insulin resistance is also associated with other diseases, including cardiovascular disease, hypertension, and polycystic ovarian syndrome. Insulin resistance is a complex condition that is not well understood. However, it is

Prompt 2: 
        Question: What are common symptoms of hypertension?
        Answer:
    
----------------------------------------------------------------------------------------------------
The most common symptoms of hypertension are headache, dizziness, and fatigue. < / FRE

,prompt_id,model_id,precision,device,system_prompt_used,decoding_strategy,max_new_tokens,prompt,generated_output,full_prompt_plus_output
0,1,microsoft/biogpt-large,bfloat16,mps,False,greedy decoding; do_sample=False; num_beams=1,96,Insulin is a hormone produced by the pancreas,that regulates blood glucose levels. Insulin i...,Insulin is a hormone produced by the pancreas ...
1,2,microsoft/biogpt-large,bfloat16,mps,False,greedy decoding; do_sample=False; num_beams=1,96,\n Question: What are common symptoms o...,The most common symptoms of hypertension are h...,Question: What are common symptoms of hyperten...
2,3,microsoft/biogpt-large,bfloat16,mps,False,greedy decoding; do_sample=False; num_beams=1,96,antibiotics are used to treat bacterial infect...,". However, the emergence of antibiotic resista...",antibiotics are used to treat bacterial infect...


## Save Baseline Outputs

The CSV file is the reusable artifact for the next comparison step. Later, after fine-tuning, we can load `baseline_outputs.csv` and compare BioGPT-Large's original answers against adapter-generated answers for the same prompts.

In [44]:
# ------------------------------------------------------------
# Save baseline outputs to CSV
# ------------------------------------------------------------

baseline_outputs_df.to_csv(BASELINE_CSV_PATH, index=False)

print(f'Saved baseline outputs CSV: {BASELINE_CSV_PATH.resolve()}')
print(f'Rows saved: {len(baseline_outputs_df)}')

# Show a clean final table for the evaluator.
display(baseline_outputs_df[['prompt_id', 'prompt', 'generated_output']])

Saved baseline outputs CSV: /Users/ymanidee/Desktop/BITS /Sem-3/My Resources/LLM For Gen AI/Assignments/Assignment-1/Code/baseline_outputs.csv
Rows saved: 3


,prompt_id,prompt,generated_output
0,1,Insulin is a hormone produced by the pancreas,that regulates blood glucose levels. Insulin i...
1,2,\n Question: What are common symptoms o...,The most common symptoms of hypertension are h...
2,3,antibiotics are used to treat bacterial infect...,". However, the emergence of antibiotic resista..."


## Submission Checklist for This Section

This section produces the required Part B Step 2 evidence:

- BioGPT-Large loaded directly from Hugging Face in BF16 precision.
- Local model path and storage footprint printed.
- Major model file sizes printed.
- Architecture report printed.
- High-level architecture tree printed.
- Decoder Layer 1 inspected in detail.
- Required weight tensor shapes printed.
- Clean architecture summary table printed.
- Exactly three biomedical baseline prompts generated without a system prompt.
- Baseline outputs saved to `baseline_outputs.csv`.